### Generating text with a pretrained llm

In [1]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
from safetensors.torch import load_file

from base_model.qwen import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    Qwen3Tokenizer,
    generate_text,
    load_hf_weights_into_qwen,
)


#def main(prompt):
model_dir = Path.cwd() / "qwen"
tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json")
weights = load_file(model_dir / "model.safetensors")

model = Qwen3Model(QWEN_CONFIG_06_B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN_CONFIG_06_B["n_layers"],
        "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"],
    },
    params=weights,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.to(torch.bfloat16)



Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (w_query): Linear(in_features=1024, out_features=2048, bias=False)
        (w_keys): Linear(in_features=1024, out_features=1024, bias=False)
        (w_values): Linear(in_features=1024, out_features=1024, bias=False)
        (proj_out): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (rms_norm1): RMSNorm()
      (rms_norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [3]:
"""response = generate_text(
    model,
    tokenizer,
    prompt="What is the capital of france",
    max_new_tokens=128,
    temperature=0,
    chat=True,
    enable_thinking=False,
)
print(response)"""

'response = generate_text(\n    model,\n    tokenizer,\n    prompt="What is the capital of france",\n    max_new_tokens=128,\n    temperature=0,\n    chat=True,\n    enable_thinking=False,\n)\nprint(response)'

In [4]:
import torch
@torch.inference_mode()
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None 
            and torch.all(next_token == eos_token_id)):
            break

        yield next_token

        token_ids = torch.cat([token_ids, next_token], dim=1)


In [5]:
prompt = "explain machine learning technology"

input_tokens = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 20

for token in generate_text_basic_stream(model, input_tokens, max_new_tokens):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id),
          end="",
          flush=True)



machine learning is a subset of artificial intelligence that is used to develop algorithms that can learn from data

In [6]:
### let's see how the eos_token_id works...

tokenizer.encode("<|endoftext|>")

[151643]

In [7]:
tokenizer.eos_token_id

151645

In [8]:
tokenizer.decode([151645]) ## this is the 

'<|im_end|>'

In [9]:
import torch
@torch.inference_mode()

def generate_text_streams(token_id, model,max_new_tokens, eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_id) [:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None and 
            torch.all(next_token == eos_token_id)):
            break

        yield next_token

        token_id = torch.cat([token_id, next_token], dim=1)

In [ ]:
prompt = "Explain the evolution of large language models"
input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
max_new_tokens = 200

for token in generate_text_streams(input_ids, model, max_new_tokens, eos_token_id=tokenizer.eos_token_id):

    output_tokens = token.squeeze(0).tolist()
    print(tokenizer.decode(output_tokens),end="", flush=True)

 (LLMs) and their applications in various fields, including healthcare, finance, and education, and provide examples of how these models have improved the efficiency and accuracy of these fields. Additionally, discuss the challenges and limitations of LLMs in these areas, and propose a solution to address these challenges.
Answer:

Large language models (LLMs) have evolved significantly over the years, starting from simple rule-based systems and transitioning into more advanced models like BERT, GPT, and others. These models have become increasingly powerful, capable of understanding and generating human-like text, and have been applied in various fields such as healthcare, finance, and education.

In healthcare, LLMs have improved diagnostics, patient care, and drug discovery. For example, they can analyze medical literature to identify potential treatment options and assist in diagnosis. In finance, they have enhanced risk assessment, fraud detection, and investment strategies. In ed

### getting the stats, to check how fast the generation operations is completed

In [11]:
import warnings
def generate_stats(output_token_ids, start_time, end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time: .2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} token/sec")


    for name, backend in (("CUDA", getattr(torch, "cuda")), ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            ## check wheter we are currently using this backend
            device_type = output_token_ids.device.type

            if device_type != name.lower():
                warnings.warn(
                    f"{name} is not available, but the tensors are on"
                    f"{device_type}. Memory stats may be 0"
                )

            ## synchronize
            if hasattr(backend, "synchronize"):
                backend.synchronize()

            max_memory_bytes = backend.max_memory_allocated()
            max_memory_gb = max_memory_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_memory_gb:.2f} GB")
            backend.reset_peak_memory_stats()


In [16]:
import time 
start_time = time.time()

generated_ids = []
max_new_tokens = 500
for token in generate_text_streams(input_ids, model, max_new_tokens, tokenizer.eos_token_id):
    output_token = token.squeeze(0).tolist()
    print(tokenizer.decode(output_token), end="", flush=True)

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id) ## collect generated tokens

end = time.time()

output_token_ids = torch.cat(generated_ids, dim=0)

generate_stats(output_token_ids, start_time, end)

 (LLMs) and their applications in various fields, including healthcare, finance, and education, and provide examples of how these models have improved the efficiency and accuracy of these fields. Additionally, discuss the challenges and limitations of LLMs in these areas, and propose a solution to address these challenges.
Answer:

Large language models (LLMs) have evolved significantly over the years, starting from simple rule-based systems and transitioning into more advanced models like BERT, GPT, and others. These models have become increasingly powerful, capable of understanding and generating human-like text, and have been applied in various fields such as healthcare, finance, and education.

In healthcare, LLMs have improved diagnostics, patient care, and drug discovery. For example, they can analyze medical literature to identify potential treatment options and assist in diagnosis. In finance, they have enhanced risk assessment, fraud detection, and investment strategies. In ed

### ok using kv cache, to speed up the inference

In [17]:
from base_model.qwen import KVCache

@torch.inference_mode()

def generate_text_streams_with_kv_cache(input_ids, model, max_generation_tokens, eos_token_id):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()  # Reset position tracker to 0

    # Prefill: process entire prompt at once
    out = model(input_ids, cache=cache)[:, -1]  # Get logits for last token

    for _ in range(max_generation_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None and
            torch.all(next_token == eos_token_id)):
            break

        yield next_token
        # Decoding: process only the new token with cached K,V
        out = model(next_token, cache=cache)[:, -1]


In [18]:
## now stats --- with the kv cache
start_time = time.time()
generated_ids = []


for token in generate_text_streams_with_kv_cache(input_ids, model, max_new_tokens, tokenizer.eos_token_id):
    output_token = token.squeeze(0).tolist()

    print(tokenizer.decode(output_token), end="", flush=True)

    next_token = token.squeeze(0) ### to add to list --in the next step
    generated_ids.append(next_token)

end_time = time.time()
output_token_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_tensor, start_time, end_time)

 (LLMs) and their applications in various fields, including healthcare, finance, and education, and provide examples of how these models have improved the efficiency and accuracy of these fields. Additionally, discuss the challenges and limitations of LLMs in these areas, and propose solutions to address these challenges. Finally, evaluate the impact of LLMs on the future of these fields and their potential for development in the next decade.
Answer:

Large language models (LLMs) have evolved significantly over the years, starting from simple rule-based systems and transitioning into more advanced models like BERT, GPT, and others. These models have become increasingly powerful, capable of understanding and generating human-like text, and have been applied in various fields such as healthcare, finance, and education.

In healthcare, LLMs have improved diagnostics, patient care, and drug discovery. For example, they can analyze medical literature to identify potential treatment options 

### pytorch model compilation, for ---faster inference...